# 05 — Hyperparameter tuning XGBoost (Optuna, Expanding Window 4 fold)

**Obiettivo.** Ottimizzare XGBoost sulle **34 feature selezionate**, valutando ogni configurazione con quattro fold temporali:

| Fold | Training | Validation |
|---|---|---|
| 2019 | 2015–2018 | 2019 |
| 2020 | 2015–2019 | 2020 |
| 2021 | 2015–2020 | 2021 |
| 2022 | 2015–2021 | 2022 |

Ogni trial usa quattro **modelli nuovi**, uno per fold, e massimizza la **media delle ROC-AUC** calcolate sulle probabilità ON/OFF. I modelli dei fold **non vengono riutilizzati** nel training definitivo: terminato il tuning, riaddestreremo il modello finale sui dati etichettati **2015–2022**.

Il dataset `feature_table_long.parquet` è prodotto dal notebook 01. **Non** vengono generate nuove feature in questo notebook.

> **Attenzione ai casi sovrapposti:** le finestre da 168 ore partono ogni giorno. Per confrontabilità con il notebook 03, di default lo split usa lo `starttime` del caso senza embargo. Se desideri una validazione più rigorosa sui confini temporali, considera un *purge* di 7 giorni; cambierà i risultati rispetto alla baseline precedente.

## 1 — Import e configurazione

`N_TRIALS_TARGET` indica **il numero totale di tentativi da raggiungere**, non il numero di nuovi tentativi ogni volta. Esempio: prima `20`, poi `50` (aggiunge i 30 mancanti), poi `100` (aggiunge altri 50). Questo è possibile grazie al database SQLite di Optuna, salvato su disco.

Se Optuna non è installato, esegui `pip install optuna` nel tuo ambiente Python prima di avviare il notebook.

In [1]:
from __future__ import annotations

import gc
import hashlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
from optuna.trial import TrialState
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# ============================================================
# TUNING CONFIGURATION — modifica soprattutto questo valore
# ============================================================
N_TRIALS_TARGET = 50      # Esempio: 20 → 50 → 100 (totale progressivo)
RANDOM_STATE = 42
N_JOBS_XGBOOST = 8        # Numero di thread di OGNI modello XGBoost
VALIDATION_YEARS = [2019, 2020, 2021, 2022]
FIRST_TRAIN_YEAR = 2015
PURGE_DAYS = 0            # 0 = split identico al notebook 03; 7 = purge sui confini

STUDY_NAME = "hydro_xgboost_expanding_4fold_34features_v1"

# Trova la root anche se Jupyter parte da Ghigu/notebooks/.
PROJECT_DIR = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / "sintef_project" / "data" / "kernel").is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Repository non trovata: apri Jupyter da ml-project-ntnu/ o da una sua sottocartella."
    )

OUTPUT_DIR = PROJECT_DIR / "Ghigu" / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_PATH = OUTPUT_DIR / "feature_table_long.parquet"

TUNING_DIR = OUTPUT_DIR / "xgboost_tuning"
TUNING_DIR.mkdir(parents=True, exist_ok=True)
STUDY_DB_PATH = TUNING_DIR / f"{STUDY_NAME}.sqlite3"
TRIALS_CSV_PATH = TUNING_DIR / f"{STUDY_NAME}_trials.csv"
BEST_PARAMS_PATH = TUNING_DIR / f"{STUDY_NAME}_best_params.json"

if N_TRIALS_TARGET < 1:
    raise ValueError("N_TRIALS_TARGET deve essere almeno 1")
if not DATASET_PATH.is_file():
    raise FileNotFoundError(f"Parquet non trovato: {DATASET_PATH}. Esegui prima il notebook 01.")

print("Dataset:", DATASET_PATH)
print("Database Optuna:", STUDY_DB_PATH)
print("Numero TOTALE di tentativi desiderato:", N_TRIALS_TARGET)

Dataset: /home/ghigu/Scrivania/ml-project-ntnu/Ghigu/data/processed/feature_table_long.parquet
Database Optuna: /home/ghigu/Scrivania/ml-project-ntnu/Ghigu/data/processed/xgboost_tuning/hydro_xgboost_expanding_4fold_34features_v1.sqlite3
Numero TOTALE di tentativi desiderato: 50


## 2 — Selezione delle feature

Queste sono **esattamente le 34 feature selezionate** nel confronto precedente: 20 base, 7 topologiche originali, 3 elettriche, 3 prezzo–water value, 1 downstream. Le altre feature presenti nel Parquet non vengono lette né usate.

In [2]:
BASE_FEATURE_COLUMNS = [
    # Posizione nella settimana e identità del generatore
    "hour", "generator", "plant",
    # Prezzo elettricità
    "price_current",
    # Parametri generatori
    "p_min", "p_max", "p_nom", "start_cost", "stop_cost", "penstock",
    # Vincoli idrici e inflow esterno
    "min_volume_Totak", "min_volume_Byrtevatn", "min_volume_Staavatn",
    "min_flow_b_Kjelavatn", "min_flow_b_Hyljelihyl", "min_flow_b_Vinjevatn",
    "min_flow_b_Byrtevatn", "min_flow_b_Bandak",
    "min_flow_r_Vest_Vassdraget", "inflow_r_Vest_Vassdraget",
]

TOPOLOGY_FEATURE_COLUMNS = [
    "direct_total_volume", "direct_mean_fill_ratio",
    "direct_total_inflow", "direct_mean_water_value",
    "upstream_total_volume", "upstream_total_inflow", "upstream_mean_water_value",
]

PRICE_FEATURE_COLUMNS = [
    "price_diff_week_mean",
    "price_rank_week",
    "price_mean_next_12h",
]

WATER_PRICE_FEATURE_COLUMNS = [
    "price_minus_water_value",
    "price_mean_week_minus_water_value",
    "price_max_week_minus_water_value",
]

EXTRA_TOPOLOGY_FEATURE_COLUMNS = [
    "downstream_water_value",
]

FEATURE_COLUMNS = (
    BASE_FEATURE_COLUMNS
    + TOPOLOGY_FEATURE_COLUMNS
    + PRICE_FEATURE_COLUMNS
    + WATER_PRICE_FEATURE_COLUMNS
    + EXTRA_TOPOLOGY_FEATURE_COLUMNS
)
ID_COLUMNS = ["run_no", "starttime", "generator", "hour"]
READ_COLUMNS = list(dict.fromkeys([*ID_COLUMNS, *FEATURE_COLUMNS, "target"]))

if len(FEATURE_COLUMNS) != 34 or len(set(FEATURE_COLUMNS)) != len(FEATURE_COLUMNS):
    raise ValueError("Attese 34 feature diverse. Controlla la selezione.")

print(f"Feature utilizzate: {len(FEATURE_COLUMNS)}")
print(f"Colonne lette dal Parquet: {len(READ_COLUMNS)}")

Feature utilizzate: 34
Colonne lette dal Parquet: 37


## 3 — Caricamento del Parquet e controlli

La conversione a `category` è importante per `generator` e `plant` (`enable_categorical=True`). Il DataFrame viene caricato **una sola volta**; nei trial si estraggono di volta in volta solo i subset richiesti per limitare la RAM.

In [3]:
df = pd.read_parquet(DATASET_PATH, columns=READ_COLUMNS)
df["starttime"] = pd.to_datetime(df["starttime"], utc=True)
for column in ["generator", "plant"]:
    df[column] = df[column].astype("category")

if df.duplicated(["run_no", "generator", "hour"]).any():
    raise ValueError("Chiavi long duplicate")
if not df["target"].notna().all() or not df["target"].isin([0, 1]).all():
    raise ValueError("Target mancante o non binario")
if df.empty:
    raise ValueError("Dataset vuoto")

AVAILABLE_YEARS = sorted(df["starttime"].dt.year.unique().tolist())
if not set(VALIDATION_YEARS).issubset(AVAILABLE_YEARS):
    raise ValueError(
        f"Mancano anni di validazione: {sorted(set(VALIDATION_YEARS)-set(AVAILABLE_YEARS))}. "
        "Controlla che il Parquet contenga tutti i casi (LIMIT_CASES=None nel notebook 01)."
    )
if FIRST_TRAIN_YEAR not in AVAILABLE_YEARS:
    raise ValueError(f"Anno iniziale {FIRST_TRAIN_YEAR} assente dal Parquet")

print(f"Dimensioni: {len(df):,} righe × {df.shape[1]} colonne lette")
print("Anni presenti:", AVAILABLE_YEARS)
print("Feature:", FEATURE_COLUMNS)
print("% ON:", f"{df['target'].mean():.2%}")

Dimensioni: 6,872,544 righe × 37 colonne lette
Anni presenti: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
Feature: ['hour', 'generator', 'plant', 'price_current', 'p_min', 'p_max', 'p_nom', 'start_cost', 'stop_cost', 'penstock', 'min_volume_Totak', 'min_volume_Byrtevatn', 'min_volume_Staavatn', 'min_flow_b_Kjelavatn', 'min_flow_b_Hyljelihyl', 'min_flow_b_Vinjevatn', 'min_flow_b_Byrtevatn', 'min_flow_b_Bandak', 'min_flow_r_Vest_Vassdraget', 'inflow_r_Vest_Vassdraget', 'direct_total_volume', 'direct_mean_fill_ratio', 'direct_total_inflow', 'direct_mean_water_value', 'upstream_total_volume', 'upstream_total_inflow', 'upstream_mean_water_value', 'price_diff_week_mean', 'price_rank_week', 'price_mean_next_12h', 'price_minus_water_value', 'price_mean_week_minus_water_value', 'price_max_week_minus_water_value', 'downstream_water_value']
% ON: 51.64%


## 4 — Definizione dei quattro fold Expanding Window

Lo split avviene sullo `starttime`: tutte le 168 ore di un caso rimangono nello stesso sottoinsieme. Ogni fold addestra un modello nuovo sui casi anteriori al proprio anno di validation. Gli indici si preparano una volta e vengono riutilizzati da Optuna.

In [4]:
case_years = df["starttime"].dt.year.to_numpy(dtype="int16")
case_starts = df["starttime"].to_numpy(dtype="datetime64[ns]")
target_array = df["target"].to_numpy(dtype="uint8")

FOLDS = []
for validation_year in VALIDATION_YEARS:
    boundary = pd.Timestamp(f"{validation_year}-01-01", tz="UTC")
    train_mask = (case_years >= FIRST_TRAIN_YEAR) & (case_years < validation_year)
    if PURGE_DAYS > 0:
        # Esclude dal train i casi iniziati troppo vicino al primo giorno di validation.
        train_mask &= case_starts < (boundary - pd.Timedelta(days=PURGE_DAYS)).to_datetime64()
    val_mask = case_years == validation_year

    train_idx = np.flatnonzero(train_mask)
    val_idx = np.flatnonzero(val_mask)
    if not len(train_idx) or not len(val_idx):
        raise ValueError(f"Fold {validation_year}: training o validazione vuoti")
    if len(np.unique(target_array[train_idx])) < 2 or len(np.unique(target_array[val_idx])) < 2:
        raise ValueError(f"Fold {validation_year}: manca una delle due classi")

    FOLDS.append({"year": validation_year, "train_idx": train_idx, "val_idx": val_idx})

    print(
        f"Fold {validation_year} | train {FIRST_TRAIN_YEAR}–{validation_year-1}: "
        f"{len(train_idx):,} righe | validation {validation_year}: {len(val_idx):,} righe "
        f"| ON valid: {target_array[val_idx].mean():.2%}"
    )

del case_years, case_starts, train_mask, val_mask, train_idx, val_idx
gc.collect()

Fold 2019 | train 2015–2018: 3,436,272 righe | validation 2019: 858,480 righe | ON valid: 51.04%
Fold 2020 | train 2015–2019: 4,294,752 righe | validation 2020: 860,832 righe | ON valid: 59.47%
Fold 2021 | train 2015–2020: 5,155,584 righe | validation 2021: 858,480 righe | ON valid: 45.54%
Fold 2022 | train 2015–2021: 6,014,064 righe | validation 2022: 858,480 righe | ON valid: 34.05%


0

## 5 — Spazio degli iperparametri e funzione `objective()`

Per ogni trial Optuna propone una configurazione e **riaddestra da zero quattro XGBoost**. La ROC-AUC è calcolata usando `predict_proba()`; nessun dato del 2023–2024 viene utilizzato.

Il **primo trial** sarà la configurazione originale del notebook 03 (`n_estimators=200`, `max_depth=5`, ecc.): in questo modo la baseline viene ricalcolata sui **quattro** fold, anziché confrontare medie calcolate su numeri di fold diversi. Non utilizziamo early stopping: anche `n_estimators` è ottimizzato direttamente.

I parametri non elencati nello spazio di ricerca rimangono fissi.

In [5]:
# Parametri fissi: non cambiano tra i fold né tra i trial
FIXED_MODEL_PARAMS = {
    "objective": "binary:logistic",
    "tree_method": "hist",
    "enable_categorical": True,
    "eval_metric": "logloss",
    "n_jobs": N_JOBS_XGBOOST,
    "random_state": RANDOM_STATE,
    "verbosity": 0,
}

# Primo trial: i parametri già utilizzati nei notebook 03/04
BASELINE_TUNABLE_PARAMS = {
    "n_estimators": 200,
    "max_depth": 5,
    "learning_rate": 0.08,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "min_child_weight": 1,
    "reg_lambda": 1.0,
}


def suggest_model_params(trial: optuna.Trial) -> dict:
    """Iperparametri che Optuna può modificare."""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.20, log=True),
        "subsample": trial.suggest_float("subsample", 0.65, 1.0, step=0.05),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0, step=0.05),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 20.0, log=True),
    }


def objective(trial: optuna.Trial) -> float:
    params = {**FIXED_MODEL_PARAMS, **suggest_model_params(trial)}
    scores = []
    start = time.perf_counter()

    for fold in FOLDS:
        train_idx, val_idx = fold["train_idx"], fold["val_idx"]
        X_train = df.loc[train_idx, FEATURE_COLUMNS].copy()
        X_val = df.loc[val_idx, FEATURE_COLUMNS].copy()
        y_train = target_array[train_idx]
        y_val = target_array[val_idx]

        model = XGBClassifier(**params)
        try:
            model.fit(X_train, y_train, verbose=False)
            probas = model.predict_proba(X_val)[:, 1]
            score = float(roc_auc_score(y_val, probas))
            scores.append(score)
        finally:
            del model, X_train, X_val, y_train, y_val
            if "probas" in locals():
                del probas
            gc.collect()

        trial.set_user_attr(f"auc_{fold['year']}", score)

    mean_auc = float(np.mean(scores))
    trial.set_user_attr("auc_mean", mean_auc)
    trial.set_user_attr("auc_std", float(np.std(scores, ddof=1)))
    trial.set_user_attr("duration_seconds", round(time.perf_counter() - start, 1))
    print(
        f"Trial {trial.number:03d} | AUC: {mean_auc:.5f} "
        f"| fold: {', '.join(f'{s:.5f}' for s in scores)}"
    )
    return mean_auc

## 6 — Optuna: database persistente e ricerca incrementale

**Come fare 20 → 50 → 100 trial:** esegui la prima volta con `N_TRIALS_TARGET = 20`. In una successiva esecuzione dello **stesso notebook**, modifica solo a `50`, poi a `100`: Optuna recupera i risultati salvati e continua senza ricominciare. Il numero indicato è sempre **totale**.

⚠️ **Non cambiare la lista di feature, i fold o lo spazio di ricerca per la stessa `STUDY_NAME`**: confrontare trial effettuati con configurazioni diverse sarebbe scorretto. Il notebook verifica una firma della configurazione e dei metadati del Parquet; per esperimenti differenti crea un nuovo `STUDY_NAME` e un nuovo database.

In [6]:
# Firma per impedire di riprendere per errore uno studio su dati o feature differenti.
# L'orario di modifica e la dimensione cambiano se rigeneri il Parquet.
file_stat = DATASET_PATH.stat()
CONFIG_TO_HASH = {
    "feature_columns": FEATURE_COLUMNS,
    "validation_years": VALIDATION_YEARS,
    "first_train_year": FIRST_TRAIN_YEAR,
    "purge_days": PURGE_DAYS,
    "dataset_path": str(DATASET_PATH.resolve()),
    "dataset_size": file_stat.st_size,
    "dataset_mtime_ns": file_stat.st_mtime_ns,
    "param_search_version": 1,
}
config_signature = hashlib.sha256(
    json.dumps(CONFIG_TO_HASH, sort_keys=True).encode("utf-8")
).hexdigest()

STORAGE_URL = f"sqlite:///{STUDY_DB_PATH.as_posix()}"
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction="maximize",
    storage=STORAGE_URL,
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.NopPruner(),
)

saved_signature = study.user_attrs.get("config_signature")
if saved_signature is None:
    if study.trials:
        raise RuntimeError("Lo studio esiste, ma non possiede una firma di configurazione: usa un nuovo STUDY_NAME.")
    study.set_user_attr("config_signature", config_signature)
    study.set_user_attr("config_info", CONFIG_TO_HASH)
    study.enqueue_trial(BASELINE_TUNABLE_PARAMS)  # Il primo trial confronta la baseline originale
elif saved_signature != config_signature:
    raise RuntimeError(
        "La configurazione o il Parquet sono cambiati rispetto ai trial già salvati. "
        "Per un nuovo esperimento cambia STUDY_NAME (e usa un DB differente), "
        "oppure ripristina il Parquet originale."
    )

# Conta i tentativi conclusi, inclusi quelli eventualmente falliti.
finished_count = sum(t.state.is_finished() for t in study.trials)
remaining_trials = max(0, N_TRIALS_TARGET - finished_count)
print(f"Trial conclusi nel database: {finished_count}")
print(f"Trial ancora da eseguire per arrivare a {N_TRIALS_TARGET}: {remaining_trials}")

if remaining_trials > 0:
    study.optimize(
        objective,
        n_trials=remaining_trials,
        n_jobs=1,   # un solo trial alla volta; XGBoost gestisce i propri thread
        gc_after_trial=True,
    )
else:
    print("Target di trial già raggiunto: nessun nuovo addestramento.")

[I 2026-10-16 13:56:24,673] Using an existing study with `study_name='hydro_xgboost_expanding_4fold_34features_v1'` instead of creating a new one.


Trial conclusi nel database: 20
Trial ancora da eseguire per arrivare a 50: 30


[I 2026-10-16 14:03:11,661] Trial 20 finished with value: 0.9451878869429896 and parameters: {'n_estimators': 700, 'max_depth': 6, 'learning_rate': 0.04400410981931022, 'subsample': 0.7000000000000001, 'colsample_bytree': 0.65, 'min_child_weight': 15, 'reg_lambda': 5.572357940488827}. Best is trial 5 with value: 0.9455676589575853.


Trial 020 | AUC: 0.94519 | fold: 0.97052, 0.90851, 0.94288, 0.95883


[I 2026-10-16 14:10:12,656] Trial 21 finished with value: 0.9437088672064541 and parameters: {'n_estimators': 700, 'max_depth': 6, 'learning_rate': 0.031344397061305714, 'subsample': 0.7000000000000001, 'colsample_bytree': 0.75, 'min_child_weight': 11, 'reg_lambda': 12.594076538203524}. Best is trial 5 with value: 0.9455676589575853.


...

Trial 036 | AUC: 0.94219 | fold: 0.96882, 0.89965, 0.94336, 0.95694


[I 2026-10-16 15:48:18,432] Trial 37 finished with value: 0.9455121124111879 and parameters: {'n_estimators': 700, 'max_depth': 4, 'learning_rate': 0.03294854653859249, 'subsample': 0.9500000000000001, 'colsample_bytree': 0.8500000000000001, 'min_child_weight': 15, 'reg_lambda': 13.561396039294939}. Best is trial 26 with value: 0.9461221741883027.


Trial 037 | AUC: 0.94551 | fold: 0.96735, 0.91722, 0.94338, 0.95410


[W 2026-10-16 15:52:32,847] Trial 38 failed with parameters: {'n_estimators': 900, 'max_depth': 4, 'learning_rate': 0.020752707185693397, 'subsample': 0.9, 'colsample_bytree': 0.75, 'min_child_weight': 13, 'reg_lambda': 5.2333658055583605} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/ghigu/Scrivania/ml-project-ntnu/Ghigu/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_109815/4064051083.py", line 52, in objective
    probas = model.predict_proba(X_val)[:, 1]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ghigu/Scrivania/ml-project-ntnu/Ghigu/.venv/lib/python3.12/site-packages/xgboost/sklearn.py", line 1972, in predict_proba
    class_probs = super().predict(
                  ^^^^^^^^^^^^^^^^
  File "/home/ghigu/Scrivania/ml-project-ntnu/Ghigu/.venv/lib/python3.12/site-packages/xgboost/core.py", 

KeyboardInterrupt: 

## 7 — Risultati, CSV e migliori parametri

La tabella contiene l'AUC media, le AUC dei quattro fold e gli iperparametri di ciascun tentativo. I migliori parametri vengono salvati anche in JSON, pronti da copiare nel notebook 03/04 o nel notebook di training definitivo. Il punteggio dei quattro fold **non** è direttamente confrontabile con il precedente `0.93841` calcolato su tre fold.

In [ ]:
completed_trials = [t for t in study.trials if t.state == TrialState.COMPLETE and t.value is not None]
if not completed_trials:
    raise RuntimeError("Nessun trial completato correttamente")

rows = []
for t in completed_trials:
    row = {
        "trial": t.number,
        "mean_roc_auc": float(t.value),
        "std_roc_auc": t.user_attrs.get("auc_std"),
        **{f"auc_{year}": t.user_attrs.get(f"auc_{year}") for year in VALIDATION_YEARS},
        "duration_seconds": t.user_attrs.get("duration_seconds"),
        **t.params,
    }
    rows.append(row)

results_df = pd.DataFrame(rows).sort_values("mean_roc_auc", ascending=False).reset_index(drop=True)
results_df.to_csv(TRIALS_CSV_PATH, index=False)

best = study.best_trial
best_model_params = {**FIXED_MODEL_PARAMS, **best.params}
report = {
    "study_name": STUDY_NAME,
    "n_completed_trials": len(completed_trials),
    "best_trial_number": best.number,
    "mean_roc_auc_4fold": float(best.value),
    "fold_scores": {str(y): best.user_attrs.get(f"auc_{y}") for y in VALIDATION_YEARS},
    "feature_columns": FEATURE_COLUMNS,
    "validation_years": VALIDATION_YEARS,
    "fixed_and_tuned_model_params": best_model_params,
}
BEST_PARAMS_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Trial COMPLETATI: {len(completed_trials)}")
print(f"Miglior trial: {best.number} | ROC-AUC media 4 fold: {best.value:.5f}")
print("AUC per fold:", report["fold_scores"])
print("\nMODEL_PARAMS da utilizzare nei successivi esperimenti:")
print(json.dumps(best_model_params, indent=2))
print("\nFile salvati:")
print(" -", STUDY_DB_PATH)
print(" -", TRIALS_CSV_PATH)
print(" -", BEST_PARAMS_PATH)

display(results_df.head(10))

## 8 — Passaggio al modello finale (dopo il tuning)

Per il modello destinato al **nuovo test 2023–2024**, usa gli iperparametri scelti e **riaddestra da zero** XGBoost su **tutti i casi etichettati 2015–2022**. Questo notebook non esegue ancora quel training finale e non legge il test della competizione.

Non sono implementati pruning o early stopping: in questa prima ricerca è preferibile confrontare tutti i quattro fold con lo stesso criterio. Il database SQLite conserva i tentativi anche se interrompi la sessione tra due trial.

**Nota metodologica:** le 34 feature sono già state selezionate guardando alcune prestazioni sul 2022. La ROC-AUC sui 4 fold è quindi una misura utile per il tuning, ma non una stima completamente indipendente; il test successivo del professore sarà il controllo esterno.